# PostgreSQL + Psycopg2 + Pandas

Neste notebook vamos integrar **Python, PostgreSQL, Psycopg2 e Pandas**.

O **Psycopg2** será usado para conectar ao PostgreSQL e executar SQL. O **Pandas** será usado para transformar os resultados em `DataFrame` e analisar os dados.

- Conexão com PostgreSQL
- `SELECT`
- PostgreSQL → Pandas
- Filtros e análises com Pandas
- `INSERT` e `UPDATE`
- Pandas → PostgreSQL
- Encerramento da conexão


## 1. Bibliotecas

- **Psycopg2** → comunicação entre Python e PostgreSQL.
- **Pandas** → manipulação e análise dos dados.

Fluxo:

`PostgreSQL → Psycopg2 → Python → Pandas DataFrame`

Se necessário:

```bash
pip install psycopg2 pandas
```


In [ ]:
# Se necessário:
# !pip install psycopg2 pandas


In [1]:
import psycopg2
import pandas as pd


## 2. Conexão com o PostgreSQL

Vamos criar **uma única conexão** para utilizar nas próximas células.

> ⚠️ Substitua os dados pelos dados do seu ambiente.


In [2]:
conn = psycopg2.connect(
    host="localhost",
    database="loja_brasil",
    user="postgres",
    password="postgres",
    port=5432
)

cur = conn.cursor()

print("Conectado ao PostgreSQL!")


Conectado ao PostgreSQL!


## 3. Executando um SELECT

O SQL continua sendo escrito normalmente. O Psycopg2 envia o comando para o PostgreSQL usando `execute()`.


In [3]:
cur.execute("""
    SELECT id_cliente, nome, cidade, email
    FROM cadastro.clientes;
""")

registros = cur.fetchall()

for registro in registros:
    print(registro)


(2, 'Bruno Henrique Souza', 'Pomerode', 'bruno.souza@email.com')
(3, 'Camila Rodrigues', 'Joinville', 'camila.rodrigues@email.com')
(4, 'Daniel Oliveira', 'Itajaí', 'daniel.oliveira@email.com')
(5, 'Eduarda Fernandes', 'Florianópolis', 'eduarda.fernandes@email.com')
(6, 'Felipe Almeida', 'Brusque', 'felipe.almeida@email.com')
(7, 'Gabriela Costa', 'Blumenau', 'gabriela.costa@email.com')
(8, 'Henrique Martins', 'Jaraguá do Sul', 'henrique.martins@email.com')
(9, 'Isabela Santos', 'Rio do Sul', 'isabela.santos@email.com')
(10, 'João Pedro Lima', 'Timbó', 'joao.lima@email.com')
(11, 'Karina Souza', 'Gaspar', 'karina.souza@email.com')
(12, 'Lucas Pereira', 'Indaial', 'lucas.pereira@email.com')
(13, 'Mariana Alves', 'Blumenau', 'mariana.alves@email.com')
(14, 'Natália Rocha', 'São José', 'natalia.rocha@email.com')
(15, 'Otávio Ribeiro', 'Chapecó', 'otavio.ribeiro@email.com')
(16, 'Patrícia Mendes', 'Blumenau', 'patricia.mendes@email.com')
(17, 'Rafael Gomes', 'Pomerode', 'rafael.gomes@email

## 4. PostgreSQL → Pandas

Agora vamos transformar o resultado do `fetchall()` em um **DataFrame**.

Esse é um dos principais objetivos da integração: consultar os dados no banco e continuar trabalhando com eles usando Pandas.


In [4]:
df_clientes = pd.DataFrame(
    registros,
    columns=["id_cliente", "nome", "cidade", "email"]
)

df_clientes


,id_cliente,nome,cidade,email
0,2,Bruno Henrique Souza,Pomerode,bruno.souza@email.com
1,3,Camila Rodrigues,Joinville,camila.rodrigues@email.com
2,4,Daniel Oliveira,Itajaí,daniel.oliveira@email.com
3,5,Eduarda Fernandes,Florianópolis,eduarda.fernandes@email.com
4,6,Felipe Almeida,Brusque,felipe.almeida@email.com
5,7,Gabriela Costa,Blumenau,gabriela.costa@email.com
6,8,Henrique Martins,Jaraguá do Sul,henrique.martins@email.com
7,9,Isabela Santos,Rio do Sul,isabela.santos@email.com
8,10,João Pedro Lima,Timbó,joao.lima@email.com
9,11,Karina Souza,Gaspar,karina.souza@email.com


### O que aconteceu?

1. `execute()` executou o SQL.
2. `fetchall()` trouxe os registros.
3. `pd.DataFrame()` transformou os registros em uma tabela Pandas.

A partir daqui podemos usar os recursos do Pandas.


In [5]:
df_clientes.head()


,id_cliente,nome,cidade,email
0,2,Bruno Henrique Souza,Pomerode,bruno.souza@email.com
1,3,Camila Rodrigues,Joinville,camila.rodrigues@email.com
2,4,Daniel Oliveira,Itajaí,daniel.oliveira@email.com
3,5,Eduarda Fernandes,Florianópolis,eduarda.fernandes@email.com
4,6,Felipe Almeida,Brusque,felipe.almeida@email.com


In [6]:
df_clientes.info()


<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id_cliente  20 non-null     int64
 1   nome        20 non-null     str  
 2   cidade      20 non-null     str  
 3   email       20 non-null     str  
dtypes: int64(1), str(3)
memory usage: 772.0 bytes


## 5. Filtrando dados com Pandas

Depois que os dados estão no DataFrame, podemos utilizar a sintaxe do Pandas.


In [7]:
df_clientes[df_clientes["cidade"] == "Florianópolis"]


,id_cliente,nome,cidade,email
3,5,Eduarda Fernandes,Florianópolis,eduarda.fernandes@email.com
19,1,Ana Paula Martins,Florianópolis,ana.martins@email.com


## 6. COUNT e GROUP BY

Também podemos deixar os cálculos para o PostgreSQL.

Neste exemplo, o banco faz o `COUNT()` e o `GROUP BY`, e o resultado é recebido pelo Pandas.


In [8]:
cur.execute("""
    SELECT cidade, COUNT(*) AS quantidade_clientes
    FROM cadastro.clientes
    GROUP BY cidade
    ORDER BY quantidade_clientes DESC;
""")

df_cidades = pd.DataFrame(
    cur.fetchall(),
    columns=["cidade", "quantidade_clientes"]
)

df_cidades


,cidade,quantidade_clientes
0,Blumenau,3
1,Itajaí,2
2,Brusque,2
3,Pomerode,2
4,Florianópolis,2
5,Balneário Camboriú,1
6,Rio do Sul,1
7,Timbó,1
8,São José,1
9,Jaraguá do Sul,1


## 7. Analisando o resultado com Pandas


In [9]:
df_cidades["quantidade_clientes"].sum()


np.int64(20)

## 8. INSERT usando Python

O Psycopg2 também pode executar comandos que alteram os dados.

Para `INSERT`, `UPDATE` e `DELETE`, usamos `commit()` para confirmar a alteração.


In [10]:
cur.execute("""
    INSERT INTO cadastro.clientes (nome, cidade, estado, email)
    VALUES ('Ana Maria', 'Blumenau', 'SC', 'ana.maria@email.com');
""")

conn.commit()

print("Cliente inserido com sucesso!")


Cliente inserido com sucesso!


## 9. Conferindo o INSERT com Pandas


In [13]:
cur.execute("""
    SELECT id_cliente, nome, cidade, email
    FROM cadastro.clientes
    WHERE email = 'ana.maria@email.com';
""")

df_novo = pd.DataFrame(
    cur.fetchall(),
    columns=["id_cliente", "nome", "cidade", "email"]
)

df_novo


,id_cliente,nome,cidade,email
0,24,Ana Maria,Itajaí,ana.maria@email.com


## 10. UPDATE usando Python

Podemos atualizar dados utilizando o mesmo cursor e a mesma conexão.


In [12]:
cur.execute("""
    UPDATE cadastro.clientes
    SET cidade = 'Itajaí'
    WHERE email = 'ana.maria@email.com';
""")

conn.commit()

print("Cliente atualizado com sucesso!")


Cliente atualizado com sucesso!


## 11. Pandas → PostgreSQL

Agora vamos fazer o caminho inverso.

Temos dados em um DataFrame e queremos enviá-los para o PostgreSQL.

Para manter o exemplo simples, vamos percorrer as linhas do DataFrame e executar um `INSERT` para cada registro.


In [14]:
df_novos_clientes = pd.DataFrame({
    "nome": ["Ana Souza", "Carlos Lima"],
    "cidade": ["Pomerode", "Joinville"],
    "estado": ["SC", "SC"],
    "email": ["ana.souza@email.com", "carlos.lima@email.com"]
})

df_novos_clientes


,nome,cidade,estado,email
0,Ana Souza,Pomerode,SC,ana.souza@email.com
1,Carlos Lima,Joinville,SC,carlos.lima@email.com


In [15]:
for _, linha in df_novos_clientes.iterrows():
    cur.execute("""
        INSERT INTO cadastro.clientes (nome, cidade, estado, email)
        VALUES (%s, %s, %s, %s);
    """, (
        linha["nome"],
        linha["cidade"],
        linha["estado"],
        linha["email"]
    ))

conn.commit()

print("Dados inseridos no PostgreSQL!")


Dados inseridos no PostgreSQL!


### Por que usamos `%s` neste exemplo?

Aqui `%s` é um **placeholder do Psycopg2** para receber valores do Python.

Ele é usado para parametrizar a consulta, em vez de montar SQL concatenando valores diretamente.

Exemplo:

```python
cur.execute(
    "INSERT INTO clientes (nome) VALUES (%s)",
    (nome,)
)
```

> O `%s` é utilizado pelo Psycopg2 para parametrizar a consulta.


## 12. Conferindo os dados inseridos


In [18]:
cur.execute("""
    SELECT id_cliente, nome, cidade, email
    FROM cadastro.clientes
    WHERE email IN (
        'ana.souza@email.com',
        'carlos.lima@email.com'
    );
""")

df_verificacao = pd.DataFrame(
    cur.fetchall(),
    columns=["id_cliente", "nome", "cidade", "email"]
)

df_verificacao


,id_cliente,nome,cidade,email
0,25,Ana Souza,Pomerode,ana.souza@email.com
1,26,Carlos Lima,Joinville,carlos.lima@email.com


## 13. SQL + Pandas: cada ferramenta no seu papel

| Ferramenta | Função |
|---|---|
| **PostgreSQL** | Armazena e consulta os dados |
| **Psycopg2** | Faz a comunicação Python ↔ PostgreSQL |
| **Pandas** | Manipula e analisa os dados |

### Fluxo

```text
Python
   │
   ▼
Psycopg2
   │ SQL
   ▼
PostgreSQL
   │ resultado
   ▼
Pandas DataFrame
   │
   ▼
Análise
```

> **Psycopg2 faz a comunicação com o banco. Pandas trabalha com os dados em Python.**


## 14. Exercício

1. Faça um `SELECT` de clientes.
2. Transforme o resultado em DataFrame.
3. Filtre clientes de uma cidade.
4. Faça um `COUNT()` por cidade.
5. Crie um DataFrame com 3 novos clientes.
6. Insira os clientes no PostgreSQL.
7. Consulte novamente para verificar.
8. Atualize um dos clientes.
9. Faça um `JOIN` entre duas tabelas e transforme o resultado em DataFrame.

### Desafio

Faça uma consulta com `JOIN`, `GROUP BY` e `COUNT()`. Depois, utilize Pandas para ordenar e filtrar o resultado.


## 15. Encerrando a conexão

Ao finalizar o uso do banco, fechamos o cursor e a conexão.


In [19]:
cur.close()
conn.close()

print("Conexão encerrada com sucesso!")


Conexão encerrada com sucesso!
